# RAG Application with LlamaIndex

## Introduction

**LlamaIndex** is a data framework for your LLM applications. It is designed to easily connect custom data sources to your LLM. While LangChain is a general-purpose framework for building LLM applications (chains, agents, etc.), LlamaIndex optimizes for **indexing** and **retrieving** data.

### Key Differences form LangChain
- **Simpler Interface**: LlamaIndex often requires less code to get a RAG system up and running.
- **Data-Centric**: It focuses heavily on the "Index" part—structuring your data for optimal retrieval.

In this notebook, we will build a RAG application to chat with our PDF documents using LlamaIndex.

## Step 1: Install Dependencies

We need to install `llama-index`.

In [ ]:
# Install llama-index
!uv pip install -q llama-index

## Step 2: Setup Environment

LlamaIndex uses OpenAI by default. We need to make sure our API key is set.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️ Warning: OPENAI_API_KEY not found. Please set it in your .env file.")
else:
    print("✅ OPENAI_API_KEY loaded")

## Step 3: Load Data

LlamaIndex makes data loading very simple with `SimpleDirectoryReader`. It automatically figures out how to parse files in a directory (PDFs, text files, etc.).

In [ ]:
from llama_index.core import SimpleDirectoryReader

# Load data from the 'pdfs' directory
reader = SimpleDirectoryReader(input_dir="pdfs")
documents = reader.load_data()

print(f"✅ Loaded {len(documents)} document pages")

## Step 4: Create Index

Now we create a `VectorStoreIndex` from our documents. This effectively:
1. Chunks the documents.
2. Embeds the chunks using OpenAI embeddings.
3. Stores them in an in-memory vector store (by default).

In [ ]:
from llama_index.core import VectorStoreIndex

# Create the index (this happens in one line!)
index = VectorStoreIndex.from_documents(documents)

print("✅ Index created")

## Step 5: Querying

To ask questions, we create a "Query Engine" from our index. This engine handles the retrieval and LLM generation loop for us.

In [ ]:
# Create a query engine
query_engine = index.as_query_engine()

# Ask a question
response = query_engine.query("What is Byte Pair Encoding?")

print("Response:")
print(response)

### Inspecting the Source

We can check which documents were used to generate the answer.

In [ ]:
for node in response.source_nodes:
    print("--------------------------------------------------")
    print(f"Score: {node.score}")
    print(f"Source File: {node.metadata.get('file_name')}")
    print(f"Content: {node.text[:200]}...")

## Conclusion

As you can see, LlamaIndex abstracts away a lot of the complexity (chunking, embedding, retrieval setup) that we explicitly handled in the LangChain version. This makes it a great choice for getting up and running quickly with standard RAG pipelines.